In [ ]:
# Step 1: Install Required Libraries
!pip install transformers datasets rouge-score kaggle

# Step 2: Disable W&B Logging
import os
os.environ["WANDB_DISABLED"] = "true"

# Step 3: Upload Kaggle API Key
from google.colab import files
files.upload()  # Upload kaggle.json here

# Step 4: Configure Kaggle and Download Dataset
!mkdir ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download gowrishankarp/newspaper-text-summarization-cnn-dailymail
!unzip -o newspaper-text-summarization-cnn-dailymail.zip -d cnn_dailymail_data

# Step 5: Load and Inspect Dataset
import pandas as pd
from datasets import Dataset

train_data = pd.read_csv("cnn_dailymail_data/cnn_dailymail/train.csv")
test_data = pd.read_csv("cnn_dailymail_data/cnn_dailymail/test.csv")
val_data = pd.read_csv("cnn_dailymail_data/cnn_dailymail/validation.csv")

# Optional: Reduce dataset size for faster training
train_data = train_data.sample(frac=0.1, random_state=42)  # Use 10% of training data
val_data = val_data.sample(frac=0.1, random_state=42)      # Use 10% of validation data

# Convert pandas DataFrames to Hugging Face Dataset format
train_dataset = Dataset.from_pandas(train_data)
val_dataset = Dataset.from_pandas(val_data)
test_dataset = Dataset.from_pandas(test_data)

# Step 6: Load PEGASUS Model and Tokenizer
from transformers import PegasusTokenizer, PegasusForConditionalGeneration

model_name = "google/pegasus-cnn_dailymail"
tokenizer = PegasusTokenizer.from_pretrained(model_name)
model = PegasusForConditionalGeneration.from_pretrained(model_name)

# Step 7: Preprocess Dataset
def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["article"], max_length=512, truncation=True, padding="max_length"
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples["highlights"], max_length=128, truncation=True, padding="max_length"
        )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = train_dataset.map(preprocess_function, batched=True)
val_dataset = val_dataset.map(preprocess_function, batched=True)

# Step 8: Define Training Arguments
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./pegasus-finetuned",
    eval_strategy="steps",
    logging_steps=50,
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    num_train_epochs=1,  # Use 1 epoch for faster training
    max_steps=1000,      # Limit training to 1000 steps
    gradient_accumulation_steps=8,
    save_steps=500,
    save_total_limit=2,
    logging_dir="./logs",
    weight_decay=0.01,
    fp16=True,  # Enable mixed precision training for speed
    report_to="none",
)

# Step 9: Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

# Step 10: Train the Model
trainer.train()

# Step 11: Save Fine-Tuned Model
model.save_pretrained("./pegasus-finetuned")
tokenizer.save_pretrained("./pegasus-finetuned")

# Step 12: Generate Summaries
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Select a subset of test articles
test_articles = test_data["article"].iloc[:10].tolist()
inputs = tokenizer(
    test_articles, max_length=512, truncation=True, padding="longest", return_tensors="pt"
)
inputs = {key: value.to(device) for key, value in inputs.items()}  # Move inputs to GPU/CPU

# Generate summaries
summary_ids = model.generate(
    inputs["input_ids"], max_length=128, num_beams=4, early_stopping=True
)
generated_summaries = tokenizer.batch_decode(summary_ids, skip_special_tokens=True)

# Step 13: Print Generated Summaries
for idx, summary in enumerate(generated_summaries):
    print(f"Generated Summary {idx + 1}: {summary}")

# Step 14: Evaluate Using ROUGE
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
references = test_data["highlights"].iloc[:10].tolist()

rouge_scores = {"rouge1": [], "rouge2": [], "rougeL": []}
for ref, pred in zip(references, generated_summaries):
    scores = scorer.score(ref, pred)
    rouge_scores["rouge1"].append(scores["rouge1"].fmeasure)
    rouge_scores["rouge2"].append(scores["rouge2"].fmeasure)
    rouge_scores["rougeL"].append(scores["rougeL"].fmeasure)

# Calculate average ROUGE scores
avg_rouge_scores = {key: sum(values) / len(values) for key, values in rouge_scores.items()}

print("Average ROUGE Scores:")
print(f"ROUGE-1: {avg_rouge_scores['rouge1']:.4f}")
print(f"ROUGE-2: {avg_rouge_scores['rouge2']:.4f}")
print(f"ROUGE-L: {avg_rouge_scores['rougeL']:.4f}")


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 14.6 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24935 sha256=bfb0a429d2a8175cba0d1618888981c529c0e9db3773291d823e4d20640c003d
  Stored in directory: /root/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge-score
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail
License(s): CC0-1.0
 99% 497M/503M [00:03<00:00, 154MB/s]
100% 503M/503M [00:03<00:00, 156MB/s]
Archive:  newspaper-text-summarization-cnn-dailymail.zip
  inflating: cnn_dailymail_data/cnn_dailymail/test.csv  
  inflating: cnn_dailymail_data/cnn_dailymail/train.csv  
  inflating: cnn_dailymail_data/cnn_dailymail/validation.csv  


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

...

Map:   0%|          | 0/1337 [00:00<?, ? examples/s]

max_steps is given, it will override any value given in num_train_epochs


Step,Training Loss,Validation Loss
50,6.651400,5.641956
100,5.832900,5.339555
150,5.552100,4.891861
200,4.928400,3.821301
250,3.685700,1.818998
300,1.923400,0.985606
350,1.010400,0.911018
400,0.751700,0.895936
450,0.724900,0.886251
500,0.683900,0.889362


/usr/local/lib/python3.10/dist-packages/transformers/modeling_utils.py:2817: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 128, 'min_length': 32, 'num_beams': 8, 'length_penalty': 0.8}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


Generated Summary 1: U.S consumer advisory group calls for 'humane treatment' of passengers . FAA conducts tests on how quickly passengers can leave a plane . Seat pitch is the distance between two seats from one point to the same point on the seat behind it is known as the pitch .
Generated Summary 2: Rahul Kumar, 17, climbed into the enclosure at Kamla Nehru Zoological Park in Ahmedabad . He ran towards the animals shouting: 'Today I kill a lion or a lion kills me!' Fortunately, he fell into a moat and was rescued by zoo security staff before reaching the animals .
Generated Summary 3: Dougie Freedman set to sign a new two-year deal at Nottingham Forest . Freedman has impressed since replacing Stuart Pearce in February . Forest owners are pleased with the job Freedman has done at the club .
Generated Summary 4: Liverpool were linked with a move for Fiorentina goalkeeper Neto in January . The Brazilian's contract expires in June and he looks set to leave the club . Neto's agent Stefan